# 3. Reproducing the results of freeze 2

**Thesis:** *Detection of Persistent Memory Poisoning in LLM Agents Using Temporal Graph Neural
Networks* (KNUST MSc).
**Benchmark:** `data/raw/ap_burst2k_nodup`, 2,000 sequences, 500 of them in the test partition.
**Configuration:** step 5 of `experiments/IMPROVEMENT.md`.
**Results:** `experiments/audit/RESULTS_FREEZE_2.md`.

This notebook reimplements nothing (the code itself is laid out in `01_benchmark.ipynb` and `02_detector.ipynb`). Every cell runs the scripts that produced the freeze, with the
same arguments, in a fresh process, as the freeze's driver (`src/experiments/coherence_plan.py`)
did. The paths and settings are imported from that driver, so the two cannot drift apart. Every
output goes to `experiments/notebook_check/`, so no frozen file is overwritten.

| Part | What it does | Time on a Quadro P2000 | Switch |
|---|---|---|---|
| 1 | Recomputes every table and figure of Chapter 4 from the saved scores of the 40 trained models (8 configurations, 5 seeds each), then compares each output with the frozen file | about 1 h, most of it bootstrap intervals and the probes | `RUN_PART_1` |
| 1b | Recomputes the two results kept from freeze 1 that Part 1 does not cover (Table 3.7, the unsupervised rows of Table 4.4) | about 2 min | `RUN_FREEZE1_ITEMS` |
| 2 | Retrains one seed of the detector from nothing (optionally also the static GAT of that seed), scores it, and compares it with the five reported seeds | about 3 to 6 h per model | `RUN_RETRAIN_DETECTOR`, `RUN_RETRAIN_STATIC` |
| 2b | Regenerates the corpus from its seeded command and compares it, array by array, with the corpus used | about 30 min and 7 GB of disk | `RUN_CORPUS_CHECK` |
| 3 | The whole process: 40 models, the online scores, the robustness sweeps and every summary | about 120 h | command only |

**What reproduces exactly, and what does not.** The summaries are deterministic given the saved
scores, so Part 1 should report every file as identical. Training on a GPU is not
bit-reproducible, so a retrained model gives slightly different scores. Part 2 therefore checks
that a retrained seed falls within the spread of the five reported seeds, not that it matches to
the last digit.

## Switches

In [ ]:
RUN_PART_1 = True              # recompute every Chapter 4 table and figure from the saved scores
RUN_FREEZE1_ITEMS = True       # Table 3.7, Table 4.4 unsupervised rows, Table 4.6 probes
RUN_RETRAIN_DETECTOR = False   # about 3 to 6 h on a Quadro P2000
RUN_RETRAIN_STATIC = False     # a further 3 h; needed only for the combined detector of that seed
RETRAIN_SEED = 1               # 1 to 5 compares with the reported model of that seed
RUN_CORPUS_CHECK = False       # about 30 min and 7 GB of disk

## Setup

In [ ]:
import filecmp, json, os, platform, shutil, subprocess, sys, time
from pathlib import Path

import numpy as np
from IPython.display import Image, Markdown, display

here = Path.cwd().resolve()
root = here if (here / "src").exists() else here.parent
sys.path.insert(0, str(root))
from src.experiments import coherence_plan as plan    # the driver that produced freeze 2

ROOT, S, R, FROZEN, DATA, SEEDS = plan.ROOT, plan.S, plan.R, plan.FROZEN, plan.DATA, plan.SEEDS
os.chdir(ROOT)
PY = sys.executable
CHECK = ROOT / "experiments" / "notebook_check"
CHECK_R = CHECK / "rebuild"
CHECK_R.mkdir(parents=True, exist_ok=True)
CONFIGS = ["tgn", "static_attn", *plan.ABLATIONS, "tgn_nocoherence"]


def git(*a):
    return subprocess.run(["git", *a], cwd=ROOT, capture_output=True, text=True).stdout.strip()


print("Repository  ", ROOT)
print("Commit      ", git("log", "-1", "--format=%h %s"))
print("Uncommitted changes under src/:", "yes" if git("status", "--porcelain", "src") else "none")
print("Python      ", platform.python_version(), sys.executable)
print("Main benchmark", DATA, "| trained runs", S.relative_to(ROOT), "| rebuild", R.relative_to(ROOT))
print("Outputs of this notebook:", CHECK.relative_to(ROOT))

In [ ]:
def run(args, log, quiet=False):
    # Run one script in a fresh process from the repository root, as the driver does.
    # Its console output goes to `log`, since several scripts print test figures as they go.
    args = [str(a) for a in args]
    t0 = time.time()
    log = Path(log)
    log.parent.mkdir(parents=True, exist_ok=True)
    with open(log, "w", encoding="utf-8") as f:
        code_ = subprocess.run(args, cwd=ROOT, stdout=f, stderr=subprocess.STDOUT).returncode
    shown = [os.path.relpath(a, ROOT) if a.startswith(str(ROOT)) else a for a in args[1:]]
    what = " ".join(a for a in shown if a not in ("-u", "-m"))[:130]
    if not quiet:
        print(f"{(time.time() - t0) / 60:6.1f} min  {what}")
    if code_:
        raise RuntimeError(f"exit code {code_}; the last lines of {log}:\n" +
                           "".join(open(log, encoding="utf-8").readlines()[-15:]))


def numbers(x, path=""):
    # Every number (and true/false) in a JSON value, with its key path. Text, such as the
    # folder a file was written to, is left out.
    if isinstance(x, dict):
        for k, v in x.items():
            yield from numbers(v, f"{path}.{k}" if path else k)
    elif isinstance(x, list):
        for i, v in enumerate(x):
            yield from numbers(v, f"{path}[{i}]")
    elif isinstance(x, (bool, int, float)):
        yield path, repr(x)


def compare(new, frozen):
    new, frozen = Path(new), Path(frozen)
    if not new.exists():
        return "NOT WRITTEN"
    if filecmp.cmp(new, frozen, shallow=False):
        return "identical"
    a, b = list(numbers(json.load(open(new)))), list(numbers(json.load(open(frozen))))
    if a == b:
        return "same numbers (only text such as a folder name differs)"
    for (pa, va), (pb, vb) in zip(a, b):
        if (pa, va) != (pb, vb):
            return f"DIFFERENT at {pa}: {va} here, {vb} frozen"
    return f"DIFFERENT: {len(a)} numbers here, {len(b)} frozen"


def table(head, rows):
    out = ["| " + " | ".join(head) + " |", "|" + "---|" * len(head)]
    out += ["| " + " | ".join(str(c) for c in r) + " |" for r in rows]
    display(Markdown("\n".join(out)))


def ci(c):
    return f"[{c[0]:.4f}, {c[1]:.4f}]"


checks = []    # (thesis item, file written here, frozen file)

## Part 1. Every Chapter 4 table and figure, recomputed from the saved scores

### 1.1 The inputs

Each of the 40 trained models wrote its scores on the validation and test partitions to
`window_scores_{val,test}_b1.npz` in its run folder. The detector and static GAT also wrote their
online scores (`online_scores_test_b1.npz`, the score after each of snapshots 9 to 19), and the
detector its scores on the duplicates benchmark (`cross_duplicates/`). The nine-feature
regression's latency scores are freeze 1's (same data, same windows).

In [ ]:
need = [S / f"{c}_s{s}" / f"window_scores_{p}_b1.npz" for c in CONFIGS for s in SEEDS for p in ("val", "test")]
need += [S / f"{m}_s{s}" / "online_scores_test_b1.npz" for m in ("tgn", "static_attn") for s in SEEDS]
need += [S / "cross_duplicates" / f"tgn_s{s}_on_dup.npz" for s in SEEDS]
need += [FROZEN / "latency_window_scores_nodup.npz", FROZEN / "latency_nodup.json",
         ROOT / DATA / "splits.json", R / "sweeps"]
missing = [p.relative_to(ROOT) for p in need if not p.exists()]
print(f"{len(need) - len(missing)} of {len(need)} inputs present")
for p in missing:
    print("missing:", p)

rows = []
for c in CONFIGS:
    hours = [json.load(open(S / f"{c}_s{s}" / "variant.json")).get("train_seconds", 0) / 3600 for s in SEEDS]
    val = [json.load(open(S / f"{c}_s{s}" / "variant.json")).get("val_window_auroc_b1") for s in SEEDS]
    rows.append([c, ", ".join(f"{v:.4f}" if v is not None else "n/a" for v in val),
                 f"{sum(hours):.1f}"])
table(["Configuration", "Validation AUROC, seeds 1 to 5 (one window = the whole sequence)",
       "Training hours, 5 seeds"], rows)

### 1.2 The summaries (the driver's phases D1 and D2)

Tables 4.1 to 4.3 and 4.5, the headline, the oracle comparison, the online drift and the
duplicates cross-score.

In [ ]:
def summary(item, name, args):
    out = CHECK / name
    run([*args, "--output", out], CHECK / (Path(name).stem + ".log"))
    checks.append((item, out, S / name))


if RUN_PART_1:
    t_part1 = time.time()
    summary("Table 4.5, headline intervals, noise floor", "summary.json",
            [PY, "-m", "src.experiments.seed_summary", "--root", S])
    summary("Tables 4.1 to 4.3, latency, validation against test", "seed_tables.json",
            [PY, "-m", "src.experiments.seed_tables", "--seeds-root", S, "--model", "tgn",
             "--data", DATA, "--online", "--lr-scores", FROZEN / "latency_window_scores_nodup.npz",
             "--lr-latency", FROZEN / "latency_nodup.json"])
    summary("Headline: combined detector, detector, static GAT", "ensemble_headline.json",
            [PY, "-m", "src.experiments.coherence_plan", "ensemble"])
    summary("Oracle comparison", "oracle_comparison_5seeds.json",
            [PY, "-m", "src.experiments.oracle_comparison", "--data", DATA,
             "--detector", *[S / f"tgn_s{s}" / "window_scores_test_b1.npz" for s in SEEDS],
             "--window-lr", FROZEN / "latency_window_scores_nodup.npz", "--coherence"])
    summary("Duplicates benchmark, cross-scored", "cross_duplicates.json",
            [PY, "-m", "src.experiments.coherence_plan", "cross-duplicates"])
    summary("Online score by snapshot", "online_drift_test.json",
            [PY, "-m", "src.experiments.online_drift", "--split", "test",
             "--runs", *[f"experiments/seeds_coherence/tgn_s{s}" for s in SEEDS]])

### 1.3 The rebuild (the driver's phase B, after the online scores)

Table 4.4 (record-level rows), Table 4.6, Figures 4.1 to 4.3 and the missed example, Tables 4.7 to
4.10 and Figures 4.4 to 4.7. The robustness sweeps themselves took about 16 GPU-hours, so their
per-seed results (`sweeps/tgn_s*.json`) are copied in and re-aggregated; Part 3 reruns them. The
attribution probes of Table 4.6 are recomputed from the corpus.

In [ ]:
if RUN_PART_1:
    shutil.copytree(R / "sweeps", CHECK_R / "sweeps", dirs_exist_ok=True)
    common = ["--data", DATA, "--seeds-root", S.relative_to(ROOT), "--out",
              CHECK_R.relative_to(ROOT), "--online"]
    rebuild = [PY, "-u", "-m", "src.experiments.rebuild"]
    for step in ("static-nodes", "record-level", "probes", "figures"):
        run([*rebuild, step, *common], CHECK_R / f"{step}.log")
    run([*rebuild, "figures", *common, "--sequence", "poisoned_placement_0463",
         "--fig-dir", "figures_missed_example"], CHECK_R / "figures_missed_example.log")
    run([*rebuild, "sweeps", *common, "--extra-sweeps", "embedding_noise_full"], CHECK_R / "sweeps.log")
    run([*rebuild, "sweep-figures", *common], CHECK_R / "sweep-figures.log")

    items = {"table_4_4_static_nodes.json": "Table 4.4, static GAT records",
             "table_4_4_record_level.json": "Table 4.4, record level",
             "probes_raw.json": "Table 4.6, the four probes (from the corpus)",
             "table_4_6_probes.json": "Table 4.6",
             "figures_4_1_to_4_3.json": "Figures 4.1 to 4.3, summary",
             "figures_missed_example_4_1_to_4_3.json": "Missed example, summary",
             "tables_4_7_to_4_10_sweeps.json": "Tables 4.7 to 4.10"}
    for name, item in items.items():
        checks.append((item, CHECK_R / name, R / name))
    for d in ("figures", "figures_missed_example"):
        for f in sorted((R / d).glob("*.json")):
            checks.append((f"Figure data, {f.stem}", CHECK_R / d / f.name, f))
    print(f"\nPart 1 took {(time.time() - t_part1) / 60:.1f} min")

### 1.4 Values kept from freeze 1

Table 3.7 (the position control), the unsupervised rows of Table 4.4. The unsupervised script also
rewrites its per-sequence scores inside the corpus folder; the notebook keeps a copy of the
original first, compares the two and puts the original back.

In [ ]:
if RUN_FREEZE1_ITEMS:
    F1 = ROOT / "experiments" / "rebuild"
    out = CHECK / "table_3_7_gate_nodup.json"
    run([PY, "-u", "-m", "src.experiments.benchmark_gate", "--data", DATA, "--split", "test",
         "--max-sequences", "500", "--output", out], CHECK / "table_3_7_gate.log")
    checks.append(("Table 3.7, position control (freeze 1)", out, F1 / "table_3_7_gate_nodup.json"))

    scores = ROOT / DATA / "unsupervised_sequence_scores_test.npz"
    kept = CHECK / "unsupervised_sequence_scores_test.original.npz"
    shutil.copy2(scores, kept)
    out = CHECK / "table_4_4_unsupervised_nodup.json"
    try:
        run([PY, "-u", "-m", "src.experiments.unsupervised_baselines", "--data", DATA,
             "--output", out], CHECK / "table_4_4_unsupervised.log")
        a, b = np.load(scores, allow_pickle=True), np.load(kept, allow_pickle=True)
        same = sorted(a.files) == sorted(b.files) and all(np.array_equal(a[k], b[k]) for k in a.files)
        print("Unsupervised per-sequence scores:", "identical to the original" if same else "DIFFERENT")
    finally:
        shutil.copy2(kept, scores)
    checks.append(("Table 4.4, unsupervised rows (freeze 1)", out, F1 / "table_4_4_unsupervised_nodup.json"))

### 1.5 The check: every output against the frozen file

In [ ]:
rows = [[compare(new, old), item, new.relative_to(ROOT).as_posix()] for item, new, old in checks]
table(["Result", "Thesis item", "Written here"], rows)
bad = [r for r in rows if not r[0].startswith(("identical", "same numbers"))]
print(f"{len(rows) - len(bad)} of {len(rows)} outputs match the frozen files" + ("" if not bad else "; see the rows marked DIFFERENT"))

### 1.6 The results, read from the files just written

Sequence level: one score per sequence, 500 test sequences (200 poisoned), 300 validation
sequences. Mean over five seeds, with the seed standard deviation and a 95 per cent bootstrap
interval.

In [ ]:
if RUN_PART_1:
    e = json.load(open(CHECK / "ensemble_headline.json"))
    names = {"detector_plus_static": "Combined detector (temporal detector + static GAT, rank mean)",
             "detector": "Temporal detector", "static": "Static GAT",
             "detector_plus_static_validation_ecdf": "Combined detector, deployable form"}
    rows = []
    for split in ("val", "test"):
        for k, label in names.items():
            if k in e[split]:
                g = e[split][k]
                rows.append([split, label, f"{g['auroc_mean']:.4f} ({g['auroc_sd']:.4f})",
                             ci(g["auroc_ci95"]), f"{g['ap_mean']:.4f}"])
    table(["Split", "Model", "AUROC mean (SD)", "95% CI", "AP"], rows)
    rows = [[k.replace("_", " "), f"{d['auroc']['delta']:+.4f}", ci(d["auroc"]["ci95"])]
            for k, d in e["test"].items() if k.endswith(("minus_detector", "minus_static"))]
    table(["Test, paired difference", "AUROC", "95% CI"], rows)

**Table 4.5, components removed** (difference = detector minus the configuration, seed-matched).

In [ ]:
if RUN_PART_1:
    s = json.load(open(CHECK / "summary.json"))
    labels = {"tgn": "Full detector",
              "no_aggregator": "Temporal aggregator removed (node memory kept)",
              "no_memory": "Node memory removed (temporal aggregator kept)",
              "static_attn": "Both removed (static GAT, attention readout)",
              "static_wide": "Both removed, widened to the detector's size",
              "meanpool": "Attention pooling replaced by mean pooling",
              "nostruct": "Structural node features removed",
              "tgn_nocoherence": "Coherence features removed"}
    rows = []
    for k, label in labels.items():
        c = s["configs"][k]
        d = c.get("reference_minus_this")
        rows.append([label, f"{c['auroc_mean']:.4f} ({c['auroc_sd']:.4f})", f"{c['ap_mean']:.4f}",
                     "reference" if d is None else f"{d['auroc']:+.4f} {ci(d['ci95'])}"])
    table(["Configuration", "AUROC mean (SD)", "AP", "Detector minus this [95% CI]"], rows)
    print(f"Noise floor (detector seed SD): {s['noise_floor_auroc_sd']:.4f}")

**Oracle comparison**: hand-built features read by a regression or gradient boosting, test partition.

In [ ]:
if RUN_PART_1:
    o = json.load(open(CHECK / "oracle_comparison_5seeds.json"))
    rows = [["Temporal detector (5 seeds)", f"{o['detector']['auroc']['mean']:.4f}", f"{o['detector']['ap']['mean']:.4f}", ""]]
    for name, r in o["oracles"].items():
        d = r.get("detector_minus_auroc")
        rows.append([name, f"{r['auroc']:.4f}", f"{r['ap']:.4f}",
                     "" if d is None else f"{d['delta']:+.4f} {ci(d['ci95'])}"])
    table(["Model", "AUROC", "AP", "Detector minus this [95% CI]"], rows)

**Table 4.1 and latency**: the detector read online, after each of snapshots 9 to 19.

In [ ]:
if RUN_PART_1:
    t = json.load(open(CHECK / "seed_tables.json"))
    rows = [[k.replace("_", " "), f"{v['mean']:.4f}" + (f" ({v['sd']:.4f})" if v.get("sd") is not None else "")]
            for k, v in t["table_4_1"]["test"].items() if isinstance(v, dict) and "mean" in v]
    table(["Test, online windows", "Mean (SD) over seeds"], rows)
    lat = t["latency"]["0.05"]
    det = lat["detector"]
    print(f"Attacks caught at a 5% false-alarm target: detector {np.mean(det['detected']):.1f} of 200 "
          f"(seeds {det['detected']}), nine-feature regression {lat['lr']['detected']} of 200")
    print(f"Detector: median snapshots to detection {np.mean(det['median']):.1f}, "
          f"realised false-alarm rate {100 * np.mean(det['test_benign_fpr']):.2f}%, "
          f"alarms before the attack {np.mean(det['pre_onset']):.3f}")
    drift = json.load(open(CHECK / "online_drift_test.json"))["mean_over_runs"]
    table(["After snapshot", *drift.keys()],
          [["AUROC (by sequence label)", *[f"{v['auroc_sequence_label']:.3f}" for v in drift.values()]]])

**Tables 4.7 to 4.10, robustness sweeps**: sequence AUROC, mean (SD) over five seeds and three draws.

In [ ]:
if RUN_PART_1:
    sw = json.load(open(CHECK_R / "tables_4_7_to_4_10_sweeps.json"))["table"]
    for name, rows_ in sw.items():
        table([name.replace("_", " "), *rows_.keys()],
              [["Sequence AUROC", *[f"{r['sequence_auroc']['mean']:.4f} ({r['sequence_auroc']['sd']:.4f})" for r in rows_.values()]]])
    dup = json.load(open(CHECK / "cross_duplicates.json"))["detector"]
    print(f"Detector trained without duplicates, scored on the duplicates benchmark: "
          f"AUROC {dup['auroc_mean']:.4f} ({dup['auroc_sd']:.4f})")

### 1.7 The figures, as just redrawn

In [ ]:
if RUN_PART_1:
    for d in ("figures", "figures_missed_example"):
        for png in sorted((CHECK_R / d).glob("*.png")):
            display(Markdown(f"**{d}/{png.name}**"))
            display(Image(filename=str(png), width=720))

## Part 2. Retrain one seed from nothing

The same command as the driver's phase A (`seed_queue` with the step 5 configuration), for one
seed, into `experiments/notebook_check/retrain/`. It trains for up to 66 epochs with early
stopping on validation, then scores the validation and test partitions. Progress is printed every
ten minutes. If the notebook is interrupted, run the cell again: finished models are skipped.

In [ ]:
RETRAIN = CHECK / "retrain"


def retrain(model):
    run_dir = RETRAIN / f"{model}_s{RETRAIN_SEED}"
    if (run_dir / "window_scores_test_b1.npz").exists():
        print(f"{run_dir.relative_to(ROOT)} already trained")
        return run_dir
    args = [PY, "-u", "-m", "src.experiments.seed_queue", "--data", DATA, "--root", RETRAIN,
            "--seeds", RETRAIN_SEED, "--models", model, *plan.CONFIG, "--coherence-features",
            "--keep-awake"]
    t0 = time.time()
    with open(RETRAIN / f"{model}_queue.log", "w", encoding="utf-8") as f:
        p = subprocess.Popen([str(a) for a in args], cwd=ROOT, stdout=f, stderr=subprocess.STDOUT)
        while p.poll() is None:
            time.sleep(600)
            log = run_dir / "train.log"
            last = [l for l in open(log, encoding="utf-8", errors="replace").read().splitlines() if l.strip()][-1:] if log.exists() else ["starting"]
            print(f"{(time.time() - t0) / 3600:5.2f} h  {last[0][:120]}", flush=True)
    if p.returncode:
        raise RuntimeError(f"training failed; see {RETRAIN / f'{model}_queue.log'}")
    print(f"{model} seed {RETRAIN_SEED} trained in {(time.time() - t0) / 3600:.2f} h")
    return run_dir


if RUN_RETRAIN_DETECTOR or RUN_RETRAIN_STATIC:
    RETRAIN.mkdir(parents=True, exist_ok=True)
    print(subprocess.run([PY, "-c", "import torch; print('torch', torch.__version__, '| GPU:', "
                          "torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')"],
                         capture_output=True, text=True).stdout)
    if RUN_RETRAIN_DETECTOR:
        retrain("tgn")
    if RUN_RETRAIN_STATIC:
        retrain("static_attn")

### 2.1 The retrained model against the reported ones

Sequence AUROC of the retrained model beside the reported model of the same seed and the range of
the five reported seeds. If both the detector and the static GAT were retrained, the combined
detector of the pair is formed the way `coherence_plan.ensemble_summary` forms it (each model's
scores turned into ranks within the partition, then averaged).

In [ ]:
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from src.experiments.duplicates_study import seq_scores

frozen = json.load(open(S / "ensemble_headline.json"))


def seq(npz, split):
    order = sorted(set(np.load(S / "tgn_s1" / f"window_scores_{split}_b1.npz", allow_pickle=True)["seq"]))
    y = np.array([int(i.startswith("poisoned")) for i in order])
    return y, seq_scores(npz, order)


rows, new = [], {}
for model, key in (("tgn", "detector"), ("static_attn", "static")):
    run_dir = RETRAIN / f"{model}_s{RETRAIN_SEED}"
    if not (run_dir / "window_scores_test_b1.npz").exists():
        continue
    for split in ("val", "test"):
        y, sc = seq(run_dir / f"window_scores_{split}_b1.npz", split)
        new[(model, split)] = (y, sc)
        rep = frozen[split][key]["auroc_per_run"]
        a = roc_auc_score(y, sc)
        lo, hi = min(rep), max(rep)
        where = "within the range" if lo <= a <= hi else f"{a - (lo if a < lo else hi):+.4f} outside the range"
        rows.append([key, split, f"{a:.4f}", f"{rep[RETRAIN_SEED - 1]:.4f}",
                     f"{lo:.4f} to {hi:.4f}", f"{np.mean(rep):.4f} ({np.std(rep, ddof=1):.4f})", where])
for split in ("val", "test"):
    if ("tgn", split) in new and ("static_attn", split) in new:
        y, d = new[("tgn", split)]
        _, t = new[("static_attn", split)]
        a = roc_auc_score(y, (rankdata(d) + rankdata(t)) / (2 * len(d)))
        rep = frozen[split]["detector_plus_static"]["auroc_per_run"]
        lo, hi = min(rep), max(rep)
        where = "within the range" if lo <= a <= hi else f"{a - (lo if a < lo else hi):+.4f} outside the range"
        rows.append(["combined", split, f"{a:.4f}", f"{rep[RETRAIN_SEED - 1]:.4f}", f"{lo:.4f} to {hi:.4f}",
                     f"{np.mean(rep):.4f} ({np.std(rep, ddof=1):.4f})", where])
if rows:
    table(["Model", "Split", f"Retrained seed {RETRAIN_SEED}", f"Reported seed {RETRAIN_SEED}",
           "Reported seeds 1 to 5", "Reported mean (SD)", "Retrained against reported"], rows)
else:
    print("Nothing retrained yet: set RUN_RETRAIN_DETECTOR = True and run Part 2.")

## Part 2b. Regenerate the corpus

The corpus is generated from a seeded command (the one in `duplicates_study.py`, step 1). This
regenerates it into `experiments/notebook_check/corpus/` and compares the partition file, the
metadata and every array of every snapshot with the corpus that was used. It does not rebuild
the training cache, which is derived from these files.

In [ ]:
if RUN_CORPUS_CHECK:
    NEW = CHECK / "corpus"
    if not (NEW / "splits.json").exists():
        run([PY, "-u", "-m", "src.data_generation.agentpoison_sequences", "--output", NEW,
             "--num-sequences", "2000", "--matched-control", "1.0", "--benign-spread", "120",
             "--attack-spread-mult", "1.0", "--seed", "42", "--no-duplicates"], CHECK / "corpus_generate.log")
    OLD = ROOT / DATA
    for name in ("splits.json", "metadata.json"):
        print(f"{name}: {compare(NEW / name, OLD / name)}")
    seqs = sorted(p.name for p in OLD.iterdir() if p.is_dir() and not p.name.startswith("_"))
    new_seqs = sorted(p.name for p in NEW.iterdir() if p.is_dir() and not p.name.startswith("_"))
    print(f"sequences: {len(new_seqs)} regenerated, {len(seqs)} used, same names: {new_seqs == seqs}")
    differ, n = [], 0
    for sq in seqs:
        for f in sorted((OLD / sq).glob("*.npz")):
            a, b = np.load(f, allow_pickle=True), np.load(NEW / sq / f.name, allow_pickle=True)
            n += 1
            if sorted(a.files) != sorted(b.files) or not all(np.array_equal(a[k], b[k]) for k in a.files):
                differ.append(f"{sq}/{f.name}")
    print(f"snapshots compared: {n}; identical in every array: {n - len(differ)}")
    for d in differ[:10]:
        print("differs:", d)

## Part 3. The whole process

The freeze was produced by one resumable command, run from the repository root in a terminal (not
in this notebook: it takes about 120 GPU-hours on a Quadro P2000, roughly five days):

```
python -m src.experiments.coherence_plan
```

It trains the 40 models (phases A and C), computes the online scores, the record-level metrics,
the probes, the figures and the robustness sweeps (phase B), and writes every summary twice from
fresh processes, stopping if the two copies differ (phases D1 and D2). Progress is written to
`experiments/seeds_coherence/PLAN.md` and `STATUS.md`. Every step skips what is already finished,
so in this repository, where every run is finished, it only re-checks. Seed 1 of the detector and
the static GAT, and seed 1 without coherence features, are the development runs of steps 4 and 5
(`experiments/option5_coherence/`, `experiments/option3_sequence/`), which the driver copies in.
Each step's command is in the driver's source, and Part 2 above is phase A for one seed.